# Locomotion Mode CNN Classifier
**Dataset:** CAMARGO (Georgia Tech, 2021) — thigh IMU + goniometer
**Classes:** walk, stair, ramp (3 classes to start)
**Goal:** Prototype CNN, retrain later on WAWA data

In [1]:
import os, glob
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
from collections import Counter
%matplotlib inline
plt.rcParams['figure.dpi'] = 130

DATA_DIR = os.path.expanduser(
    '~/repos/projects/exo-assist-pipeline/data/camargo/AB06/10_09_18')
print('Modes:', [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR,d))])

Modes: ['levelground', 'treadmill', 'stair', 'static', 'ramp']


## 1. Inspect .mat structure

In [11]:
import pandas as pd
df = pd.read_csv(os.path.expanduser(
    '~/repos/projects/exo-assist-pipeline/data/camargo/AB06/10_09_18/levelground/imu/levelground_ccw_fast_01_01.csv'))
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

Shape: (3068, 25)
Columns: ['Header', 'foot_Accel_X', 'foot_Accel_Y', 'foot_Accel_Z', 'foot_Gyro_X', 'foot_Gyro_Y', 'foot_Gyro_Z', 'shank_Accel_X', 'shank_Accel_Y', 'shank_Accel_Z', 'shank_Gyro_X', 'shank_Gyro_Y', 'shank_Gyro_Z', 'thigh_Accel_X', 'thigh_Accel_Y', 'thigh_Accel_Z', 'thigh_Gyro_X', 'thigh_Gyro_Y', 'thigh_Gyro_Z', 'trunk_Accel_X', 'trunk_Accel_Y', 'trunk_Accel_Z', 'trunk_Gyro_X', 'trunk_Gyro_Y', 'trunk_Gyro_Z']


,Header,foot_Accel_X,foot_Accel_Y,foot_Accel_Z,foot_Gyro_X,foot_Gyro_Y,foot_Gyro_Z,shank_Accel_X,shank_Accel_Y,shank_Accel_Z,...,thigh_Accel_Z,thigh_Gyro_X,thigh_Gyro_Y,thigh_Gyro_Z,trunk_Accel_X,trunk_Accel_Y,trunk_Accel_Z,trunk_Gyro_X,trunk_Gyro_Y,trunk_Gyro_Z
0,14.420,-0.714549,0.064110,0.658318,-0.871286,6.930115,-0.749221,-1.522151,-0.121209,-0.181665,...,0.007293,-0.894220,0.812964,1.130518,-1.289296,-0.092787,-0.267498,-0.454489,-0.118655,-0.481277
1,14.425,-0.900126,-0.022291,0.595152,-0.813898,7.216554,-0.860604,-3.274438,-0.509360,-0.566609,...,0.007581,-0.907899,0.844110,1.261977,-1.275340,-0.088247,-0.330486,-0.475608,-0.068503,-0.397365
2,14.430,-1.125202,-0.089129,0.745914,-0.699878,7.508803,-0.946267,-0.701983,-0.375555,-1.730934,...,0.011200,-0.847414,0.875439,1.409149,-1.195575,-0.094413,-0.369659,-0.570394,-0.109185,-0.354071
3,14.435,-1.191269,-0.011599,0.935981,-0.523642,7.589205,-0.932658,-0.470476,0.001592,-0.983555,...,-0.004236,-0.670965,0.888611,1.519886,-1.110505,-0.095950,-0.387671,-0.751381,-0.236083,-0.308336
4,14.440,-1.226409,0.141018,1.109809,-0.271811,7.550313,-0.857894,-1.329775,0.240656,-0.691021,...,-0.024356,-0.518411,0.889074,1.657576,-1.053306,-0.099222,-0.364193,-0.940229,-0.430369,-0.303552


In [2]:
sample = glob.glob(os.path.join(DATA_DIR, 'levelground', 'imu', '*.mat'))[0]
print(f'File: {os.path.basename(sample)}')
d = sio.loadmat(sample)
for k, v in d.items():
    if not k.startswith('__'):
        print(f'  {k}: shape={v.shape}, dtype={v.dtype}')

File: levelground_ccw_slow_03_01.mat
  None: shape=(1,), dtype=[('s0', 'O'), ('s1', 'O'), ('s2', 'O'), ('arr', 'O')]


In [3]:
for sensor in ['gon', 'conditions']:
    files = glob.glob(os.path.join(DATA_DIR, 'levelground', sensor, '*.mat'))
    if files:
        d2 = sio.loadmat(files[0])
        print(f'\n=== {sensor} ===')
        for k, v in d2.items():
            if not k.startswith('__'):
                print(f'  {k}: shape={getattr(v,"shape","?")}')


=== gon ===
  None: shape=(1,)

=== conditions ===
  subject: shape=(0,)
  file: shape=(1,)
  trialType: shape=(1,)
  leadingLegStart: shape=(1, 1)
  leadingLegStop: shape=(1, 1)
  labelError: shape=(1, 1)
  trialStarts: shape=(1, 1)
  trialEnds: shape=(1, 1)
  turn: shape=(1,)
  speed: shape=(1,)
  None: shape=(1,)


## 2. Config & Load
CAMARGO IMU layout (24 cols, 200 Hz): foot(0-5), shank(6-11), **thigh(12-17)**, trunk(18-23).
**Update THIGH_COLS after running cell 1!**

In [13]:
# === COMPLETE CSV DATA LOADER ===
import pandas as pd

DATA_DIR = os.path.expanduser(
    '~/repos/projects/exo-assist-pipeline/data/camargo/AB06/10_09_18')

THIGH_COLS = ['thigh_Accel_X', 'thigh_Accel_Y', 'thigh_Accel_Z',
              'thigh_Gyro_X', 'thigh_Gyro_Y', 'thigh_Gyro_Z']
FS = 200
WIN = 40    # 200ms
STEP = 20   # 100ms
LABEL_MAP = {'levelground': 0, 'ramp': 1, 'stair': 2}
LABELS = ['walk', 'ramp', 'stair']

# Load all trials
trials = []
for mode in ['levelground', 'ramp', 'stair']:
    csv_files = sorted(glob.glob(os.path.join(DATA_DIR, mode, 'imu', '*.csv')))
    for f in csv_files:
        df = pd.read_csv(f)
        data = df[THIGH_COLS].values.astype(np.float32)
        trials.append((data, mode, os.path.basename(f)))

print(f'{len(trials)} trials: {Counter([t[1] for t in trials])}')
print(f'Sample shape: {trials[0][0].shape}')  # (n_samples, 6)

# Create sliding windows
Xw, yw = [], []
for data, mode, _ in trials:
    lab = LABEL_MAP[mode]
    for s in range(0, data.shape[0] - WIN + 1, STEP):
        Xw.append(data[s:s+WIN].T)  # (6, 40) for Conv1d
        yw.append(lab)

X = np.array(Xw, dtype=np.float32)
y = np.array(yw, dtype=np.int64)
print(f'\nX: {X.shape}  y: {y.shape}')
print({LABELS[i]: c for i, c in enumerate(np.bincount(y))})

# Normalize
Xm = X.mean(axis=(0, 2), keepdims=True)
Xs = X.std(axis=(0, 2), keepdims=True) + 1e-8
X = (X - Xm) / Xs

# Split
from sklearn.model_selection import train_test_split
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {Xtr.shape}, Test: {Xte.shape}')

135 trials: Counter({'ramp': 63, 'stair': 42, 'levelground': 30})
Sample shape: (3068, 6)

X: (23202, 6, 40)  y: (23202,)
{'walk': np.int64(6126), 'ramp': np.int64(10788), 'stair': np.int64(6288)}
Train: (18561, 6, 40), Test: (4641, 6, 40)


In [5]:
def load_imu(path):
    d = sio.loadmat(path)
    for k, v in d.items():
        if not k.startswith('__') and hasattr(v,'shape') and len(v.shape)==2 and v.shape[0]>10:
            return v.astype(np.float32)
    raise ValueError(f'No data in {path}')

trials = []
for mode in ['levelground', 'ramp', 'stair']:
    for f in sorted(glob.glob(os.path.join(DATA_DIR, mode, 'imu', '*.mat'))):
        try: trials.append((load_imu(f), mode, os.path.basename(f)))
        except Exception as e: print(f'Error: {e}')

print(f'{len(trials)} trials: {Counter([t[1] for t in trials])}')
print(f'Shape: {trials[0][0].shape}')

Error: No data in /home/wxp/repos/projects/exo-assist-pipeline/data/camargo/AB06/10_09_18/levelground/imu/levelground_ccw_fast_01_01.mat
Error: No data in /home/wxp/repos/projects/exo-assist-pipeline/data/camargo/AB06/10_09_18/levelground/imu/levelground_ccw_fast_02_01.mat
Error: No data in /home/wxp/repos/projects/exo-assist-pipeline/data/camargo/AB06/10_09_18/levelground/imu/levelground_ccw_fast_03_01.mat
Error: No data in /home/wxp/repos/projects/exo-assist-pipeline/data/camargo/AB06/10_09_18/levelground/imu/levelground_ccw_fast_04_01.mat
Error: No data in /home/wxp/repos/projects/exo-assist-pipeline/data/camargo/AB06/10_09_18/levelground/imu/levelground_ccw_fast_05_01.mat
Error: No data in /home/wxp/repos/projects/exo-assist-pipeline/data/camargo/AB06/10_09_18/levelground/imu/levelground_ccw_normal_01_01.mat
Error: No data in /home/wxp/repos/projects/exo-assist-pipeline/data/camargo/AB06/10_09_18/levelground/imu/levelground_ccw_normal_02_01.mat
Error: No data in /home/wxp/repos/pro

IndexError: list index out of range

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 7))
for ax, mode in zip(axes, ['levelground', 'ramp', 'stair']):
    trial = next(t for t in trials if t[1] == mode)
    data = trial[0][:, THIGH_COLS]
    for ch in range(data.shape[1]):
        ax.plot(np.arange(data.shape[0])/FS, data[:,ch], lw=0.5, alpha=0.7)
    ax.set_title(mode, fontweight='bold'); ax.set_ylabel('IMU'); ax.grid(alpha=0.3)
axes[-1].set_xlabel('Time (s)')
fig.tight_layout(); plt.show()

## 3. Sliding windows

In [ ]:
Xw, yw = [], []
for data, mode, _ in trials:
    thigh = data[:, THIGH_COLS]
    lab = LABEL_MAP[mode]
    for s in range(0, thigh.shape[0]-WIN+1, STEP):
        Xw.append(thigh[s:s+WIN].T); yw.append(lab)
X = np.array(Xw, dtype=np.float32)
y = np.array(yw, dtype=np.int64)
print(f'X: {X.shape}  y: {y.shape}')
print({LABELS[i]: c for i,c in enumerate(np.bincount(y))})

In [ ]:
Xm = X.mean(axis=(0,2), keepdims=True)
Xs = X.std(axis=(0,2), keepdims=True) + 1e-8
X = (X - Xm) / Xs

from sklearn.model_selection import train_test_split
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {Xtr.shape}, Test: {Xte.shape}')

## 4. CNN model

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {dev}')

class CNN(nn.Module):
    def __init__(self, nch=6, nc=3):
        super().__init__()
        self.feat = nn.Sequential(
            nn.Conv1d(nch,32,5,padding=2), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32,64,5,padding=2), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64,128,3,padding=1), nn.BatchNorm1d(128), nn.ReLU(), nn.AdaptiveAvgPool1d(1))
        self.head = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(128,64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64,nc))
    def forward(self, x): return self.head(self.feat(x).squeeze(-1))

model = CNN(N_CH, NC).to(dev)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

## 5. Train

In [ ]:
trdl = DataLoader(TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr)), 64, shuffle=True)
tedl = DataLoader(TensorDataset(torch.from_numpy(Xte), torch.from_numpy(yte)), 128)
w = torch.FloatTensor(len(np.bincount(ytr))/(len(np.bincount(ytr))*np.bincount(ytr))).to(dev)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
lf = nn.CrossEntropyLoss(weight=w)
sc = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)

H = {'tl':[], 'vl':[], 'va':[]}
for ep in range(50):
    model.train(); tl=0
    for xb,yb in trdl:
        xb,yb = xb.to(dev), yb.to(dev)
        opt.zero_grad(); loss=lf(model(xb),yb); loss.backward(); opt.step()
        tl += loss.item()*len(xb)
    tl /= len(trdl.dataset)
    model.eval(); vl=cor=0
    with torch.no_grad():
        for xb,yb in tedl:
            xb,yb = xb.to(dev), yb.to(dev)
            out=model(xb); vl+=lf(out,yb).item()*len(xb); cor+=(out.argmax(1)==yb).sum().item()
    vl/=len(tedl.dataset); va=cor/len(tedl.dataset); sc.step(vl)
    H['tl'].append(tl); H['vl'].append(vl); H['va'].append(va)
    if (ep+1)%5==0 or ep==0: print(f'Ep {ep+1:3d}  loss={tl:.4f}/{vl:.4f}  acc={va:.3f}')
print(f'\nBest: {max(H["va"]):.3f}')

In [ ]:
fig,(a1,a2) = plt.subplots(1,2,figsize=(11,4))
a1.plot(H['tl'],label='Train'); a1.plot(H['vl'],label='Test'); a1.legend(); a1.grid(alpha=0.3)
a1.set_xlabel('Epoch'); a1.set_ylabel('Loss')
a2.plot(H['va'],color='green'); a2.grid(alpha=0.3)
a2.set_xlabel('Epoch'); a2.set_ylabel('Acc'); a2.set_title(f'Best: {max(H["va"]):.3f}')
fig.tight_layout(); plt.show()

## 6. Confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
model.eval(); ps,ls = [],[]
with torch.no_grad():
    for xb,yb in tedl:
        ps.extend(model(xb.to(dev)).argmax(1).cpu().numpy()); ls.extend(yb.numpy())
print(classification_report(ls, ps, target_names=LABELS))
cm = confusion_matrix(ls, ps)
fig,ax = plt.subplots(figsize=(5,4))
ax.imshow(cm, cmap='Blues')
for i in range(NC):
    for j in range(NC):
        ax.text(j,i,str(cm[i,j]),ha='center',va='center',
                color='white' if cm[i,j]>cm.max()/2 else 'black')
ax.set_xticks(range(NC)); ax.set_yticks(range(NC))
ax.set_xticklabels(LABELS); ax.set_yticklabels(LABELS)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title('Confusion Matrix')
fig.tight_layout(); plt.show()

## 7. Save

In [ ]:
sd = os.path.expanduser('~/repos/projects/exo-assist-pipeline/models')
os.makedirs(sd, exist_ok=True)
torch.save({'state_dict':model.state_dict(), 'config':{'n_ch':N_CH,'nc':NC,'win':WIN,'fs':FS},
    'labels':LABELS, 'norm':{'mean':Xm,'std':Xs}, 'acc':max(H['va'])},
    os.path.join(sd,'locomotion_cnn_v1.pt'))
print('Saved!')

## Next steps
1. Run cell 1, update `THIGH_COLS` if needed
2. Add gon (hip angle) channels → 10-ch input
3. Split ramp/stair into ascent/descent → 5 classes
4. More subjects for cross-subject eval
5. Collect + swap in WAWA data (tailbone IMU + motor encoders + EMG)